In [1]:
# Cell 1: Install core dependencies
%pip install python-dotenv langchain langchain-openai trafilatura -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 2: Load environment variables and verify keys are present
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env"
assert SERPER_API_KEY, "SERPER_API_KEY not found in .env"

print(f"OpenAI model : {OPENAI_MODEL}")
print(f"OpenAI key   : ...{OPENAI_API_KEY[-6:]}")
print(f"Serper key   : ...{SERPER_API_KEY[-6:]}")

OpenAI model : gpt-4o-mini
OpenAI key   : ...i-lLAA
Serper key   : ...d053d8


In [3]:
# Cell 2b: Disk-based cache for Serper and LLM calls
# Avoids re-calling paid APIs when re-running notebook cells.
# Cache lives in .cache/ as JSON files keyed by SHA-256 of the input.
import hashlib, json, pathlib

CACHE_DIR = pathlib.Path(".cache")
CACHE_DIR.mkdir(exist_ok=True)

def _cache_key(*parts) -> str:
    """Create a deterministic hash from arbitrary string parts."""
    raw = json.dumps(parts, sort_keys=True, ensure_ascii=True)
    return hashlib.sha256(raw.encode()).hexdigest()

def cache_get(namespace: str, *key_parts):
    """Return cached value or None if miss."""
    h = _cache_key(*key_parts)
    p = CACHE_DIR / namespace / f"{h}.json"
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return None

def cache_set(namespace: str, value, *key_parts):
    """Write value to cache."""
    h = _cache_key(*key_parts)
    d = CACHE_DIR / namespace
    d.mkdir(exist_ok=True)
    p = d / f"{h}.json"
    p.write_text(json.dumps(value, ensure_ascii=False, indent=1), encoding="utf-8")

def cache_stats():
    """Print cache stats per namespace."""
    if not CACHE_DIR.exists():
        print("Cache: empty")
        return
    for ns in sorted(CACHE_DIR.iterdir()):
        if ns.is_dir():
            files = list(ns.glob("*.json"))
            print(f"  {ns.name}: {len(files)} entries")

print(f"Cache dir: {CACHE_DIR.resolve()}")
cache_stats()

Cache dir: C:\Users\Abhishek A\Defining_Category\.cache
  llm_scoring: 25 entries
  serper: 30 entries


In [4]:
# Cell 3: Category config + comprehensive trusted source registry (per doc1 §3 + doc2)
# Every domain from doc2 is included, organized by tier for search prioritization

TEST_CATEGORY = "Account-Based Marketing"

# doc1 §3: maturity tag — determines currency thresholds for scoring
# "emerging" = 6-12 months, "evolving" = 18-24 months, "stable" = 36+ months
CATEGORY_MATURITY = "evolving"  # ABM is established but still shifting (ABM→ABX drift)

# doc1 §3: gather under ALL aliases
CATEGORY_ALIASES = [
    "Account-Based Marketing",
    "ABM",
    "Account-Based Marketing Platforms",
    "ABM platforms",
    "Account-Based Everything",
    "ABX",
    "Account-Based Experience",
]

# ── Trusted sites from doc2, organized by tier ──────────────────────────

# TIER 1: Major industry analysts — highest value, search individually/small batches
# NOTE: Google site: operator is unreliable with deep sub-paths.
# Use root domains where safe. Forrester is clean per doc2 —
# "no major review-platform subdivision to filter out."
# Gartner needs care (reviews/digital-markets excluded via DROP_URL_PATTERNS).
TIER1_SITES = [
    # Gartner — root subdomains + glossary paths
    "blogs.gartner.com",
    "gartner.com/en/articles",
    "gartner.com/en/marketing/glossary",
    "gartner.com/en/information-technology/glossary",
    "gartner.com/en/sales/glossary",
    # Forrester — root domain is safe (doc2: "unusually clean")
    "forrester.com",
    "go.forrester.com",
    # IDC — root domain + blog subdomain
    "idc.com",
    "blogs.idc.com",
]

# TIER 2: Independent analysts — often more open access
TIER2_SITES = [
    "constellationr.com",
    "infotech.com",
    "451research.com",
    "spglobal.com/marketintelligence",
    "omdia.tech.informa.com",
    "hfsresearch.com",
    "isg-one.com",
    "everestgrp.com",
    "nucleusresearch.com",
    "dresneradvisory.com",
    "abiresearch.com",
    "gigaom.com",
    "aragonresearch.com",
    "moorinsightsstrategy.com",
    "pund-it.com",
    "enderlegroup.com",
    "jgoldassociates.com",
]

# TIER 2b: Domain-specialist analysts
TIER2B_SITES = [
    "kuppingercole.com",
    "barc.com",
    "frost.com",
    "enterprisemanagement.com",   # EMA
    "esg-global.com",
    "tag-cyber.com",
    "securosis.com",
    "colemanparkes.com",
]

# TIER 3: Trade publications (byline-level filter applies downstream in scoring)
TRADE_PUB_SITES = [
    # Martech-specific
    "chiefmartec.com",
    "martech.org",
    "adexchanger.com",
    "digiday.com",
    # TechTarget properties
    "searchcrm.com",
    "searchsecurity.com",
    "searchdatamanagement.com",
    # IDG family
    "cio.com",
    "computerworld.com",
    "infoworld.com",
    "csoonline.com",
    # Informa family
    "darkreading.com",
    "informationweek.com",
    # Independent
    "diginomica.com",
    "theregister.com",
    "zdnet.com",
    # Named SME blogs
    "stratechery.com",
    "tomtunguz.com",
    "ben-evans.com",
]

# TIER 4: Practitioner/consultancy publications
CONSULTANCY_SITES = [
    "mckinsey.com",
    "bcg.com",
    "bain.com",
    "deloitte.com/insights",
    "accenture.com",
    "ey.com",
    "kpmg.com",
    "a16z.com",
]

# TIER 5: Academic / standards
ACADEMIC_SITES = [
    "hbr.org",
    "sloanreview.mit.edu",
    "nist.gov",
]

# Combine all for reference
ALL_TRUSTED_SITES = (TIER1_SITES + TIER2_SITES + TIER2B_SITES
                     + TRADE_PUB_SITES + CONSULTANCY_SITES + ACADEMIC_SITES)

# ── doc1 §4: Known analyst author-hub URLs for this category ────────────
# These are direct crawl targets — single-URL goldmines per doc1.
ANALYST_HUB_URLS = [
    # Forrester ABM analysts
    "https://www.forrester.com/blogs/author/john_arnold/",
    "https://www.forrester.com/blogs/author/jessie_johnson/",
    "https://www.forrester.com/blogs/author/terry_flaherty/",
    # Gartner articles (curated, includes TOPO acquisitions)
    "https://www.gartner.com/en/articles/the-account-based-everything-framework",
    # ISG / Ventana Research — Keith Dawson on ABM
    "https://research.isg-one.com/analyst-perspectives/topic/intelligent-marketing",
    # Scott Brinker — martech landscape
    "https://chiefmartec.com/category/account-based-marketing/",
]

# ── Explicit DROP patterns (per doc2 "What's deliberately not on this list") ──
DROP_URL_PATTERNS = [
    "/software-reviews/",       # Info-Tech SoftwareReviews = review platform
    "/compare/",                # head-to-head comparison pages
    "/products/",               # product review pages
    "gpivendorresources",       # Gartner vendor portal
    "gartner.com/reviews",      # Gartner Peer Insights
    "gartner.com/en/digital-markets",  # Capterra/GetApp/Software Advice
    "g2.com", "trustradius.com", "capterra.com", "getapp.com",
    "sourceforge.net", "goodfirms.co", "crozdesk.com",
    # Event/sponsor pages (low content value)
    "/sponsors/",
    "/event/",
]

# ── Google-operator exclusions (applied IN the Serper query itself) ──────
# Makes Google filter junk BEFORE returning results, saving API credits.
SERPER_EXCLUDE_SITES = [
    "store.frost.com",              # Frost paywall store
    "my.idc.com",                   # IDC paywalled docs
    "info.idc.com",                 # IDC gated lead-gen content
    "view.frost.com",               # Frost gated viewer
    "hub.frost.com",                # Frost hub pages
    "web-assets.bcg.com",           # BCG PDF/asset CDN
    "keithdawson.isg-one.com",      # personal blog subdomain
    "portal.gigaom.com",            # GigaOm paywalled portal
]
SERPER_EXCLUDE_INURL = [
    "/wp-content/uploads/",         # WordPress uploaded PDFs/images
    "/content/dam/",                # CMS asset dirs (Accenture, Deloitte)
    "/docs/default-source/",        # ISG document library
    "/downloads/",                  # IDC/misc download dirs
    "event-pdf-generator",          # Forrester event PDFs
]

# Build the exclusion string once — appended to every Serper query
_exc_parts = ["-filetype:pdf"]
_exc_parts += [f"-site:{s}" for s in SERPER_EXCLUDE_SITES]
_exc_parts += [f'-inurl:"{p}"' for p in SERPER_EXCLUDE_INURL]
SERPER_EXCLUSIONS = " ".join(_exc_parts)

# ── Currency thresholds (months) based on maturity ──────────────────────
CURRENCY_THRESHOLDS = {"emerging": 12, "evolving": 24, "stable": 36}
MAX_SOURCE_AGE_MONTHS = CURRENCY_THRESHOLDS[CATEGORY_MATURITY]

print(f"Category      : {TEST_CATEGORY}")
print(f"Maturity      : {CATEGORY_MATURITY} (max source age: {MAX_SOURCE_AGE_MONTHS} months)")
print(f"Aliases       : {len(CATEGORY_ALIASES)}")
print(f"Tier 1 sites  : {len(TIER1_SITES)} (Gartner, Forrester, IDC)")
print(f"Tier 2 sites  : {len(TIER2_SITES)} (independent analysts)")
print(f"Tier 2b sites : {len(TIER2B_SITES)} (domain specialists)")
print(f"Trade pubs    : {len(TRADE_PUB_SITES)}")
print(f"Consultancies : {len(CONSULTANCY_SITES)}")
print(f"Academic      : {len(ACADEMIC_SITES)}")
print(f"Total sites   : {len(ALL_TRUSTED_SITES)}")
print(f"Analyst hubs  : {len(ANALYST_HUB_URLS)}")
print(f"Drop patterns : {len(DROP_URL_PATTERNS)}")
print(f"Serper excl.  : {len(SERPER_EXCLUDE_SITES)} sites, {len(SERPER_EXCLUDE_INURL)} inurl, +pdf filter")
print(f"\nExclusion string ({len(SERPER_EXCLUSIONS)} chars):")
print(f"  {SERPER_EXCLUSIONS}")

Category      : Account-Based Marketing
Maturity      : evolving (max source age: 24 months)
Aliases       : 7
Tier 1 sites  : 9 (Gartner, Forrester, IDC)
Tier 2 sites  : 17 (independent analysts)
Tier 2b sites : 8 (domain specialists)
Trade pubs    : 19
Consultancies : 8
Academic      : 3
Total sites   : 64
Analyst hubs  : 6
Drop patterns : 15
Serper excl.  : 8 sites, 5 inurl, +pdf filter

Exclusion string (325 chars):
  -filetype:pdf -site:store.frost.com -site:my.idc.com -site:info.idc.com -site:view.frost.com -site:hub.frost.com -site:web-assets.bcg.com -site:keithdawson.isg-one.com -site:portal.gigaom.com -inurl:"/wp-content/uploads/" -inurl:"/content/dam/" -inurl:"/docs/default-source/" -inurl:"/downloads/" -inurl:"event-pdf-generator"


In [ ]:
# Cell 4: Focused Serper search — Tier 1 + key Tier 2 only
# Target ~50 URLs by adding secondary aliases for Tier 1 and increasing num_per_query.
import requests, time

SEARCH_DELAY = 0.3  # seconds between Serper calls

def serper_search(query: str, api_key: str, num: int = 10, **kwargs) -> list[dict]:
    """Call Serper.dev Google Search API. Cached to disk."""
    cached = cache_get("serper", query, num, kwargs)
    if cached is not None:
        return cached
    payload = {"q": query, "num": num}
    payload.update(kwargs)
    resp = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
        json=payload,
        timeout=15,
    )
    resp.raise_for_status()
    results = resp.json().get("organic", [])
    cache_set("serper", results, query, num, kwargs)
    time.sleep(SEARCH_DELAY)
    return results

def batch_site_queries(sites: list[str], batch_size: int = 5) -> list[str]:
    batches = []
    for i in range(0, len(sites), batch_size):
        chunk = sites[i : i + batch_size]
        clause = " OR ".join(f"site:{s}" for s in chunk)
        batches.append(f"({clause})")
    return batches

def url_is_blocked(url: str) -> bool:
    url_lower = url.lower()
    for pattern in DROP_URL_PATTERNS:
        if pattern in url_lower:
            return True
    return False

def run_search_pass(name: str, sites: list[str], aliases: list[str],
                    batch_size: int, seen: set, results: list,
                    num_per_query: int = 6):
    """Run a search pass. Appends SERPER_EXCLUSIONS to every query."""
    batches = batch_site_queries(sites, batch_size=batch_size)
    queries_run = 0
    hits_added = 0
    blocked = 0
    cache_hits = 0
    for alias in aliases:
        for site_clause in batches:
            query = f'{site_clause} "{alias}" {SERPER_EXCLUSIONS}'
            queries_run += 1
            try:
                was_cached = cache_get("serper", query, num_per_query, {}) is not None
                if was_cached:
                    cache_hits += 1
                hits = serper_search(query, SERPER_API_KEY, num=num_per_query)
                for h in hits:
                    url = h.get("link", "")
                    if not url or url in seen:
                        continue
                    if url_is_blocked(url):
                        blocked += 1
                        continue
                    seen.add(url)
                    results.append({
                        "url": url,
                        "title": h.get("title", ""),
                        "snippet": h.get("snippet", ""),
                        "query_alias": alias,
                        "search_pass": name,
                    })
                    hits_added += 1
            except Exception as e:
                print(f"    ✗ query failed: {e}")
    cached_msg = f", {cache_hits} from cache" if cache_hits else ""
    print(f"  {name}: {queries_run} queries → {hits_added} new URLs (blocked {blocked}{cached_msg})")
    return queries_run

# ── Focused aliases ──
FOCUSED_ALIASES = [
    "Account-Based Marketing",
    "ABM platforms",
]

# Secondary aliases for Tier 1 only (higher precision)
SECONDARY_ALIASES = [
    "Account-Based Marketing Platforms",
    "Account-Based Everything",
]

# ── Key Tier 2 sites (martech/ABM relevant only) ──
KEY_TIER2_SITES = [
    "constellationr.com",
    "isg-one.com",          # includes former Ventana Research
    "gigaom.com",
    "nucleusresearch.com",
    "aragonresearch.com",
]

seen_urls = set()
all_results = []
total_queries = 0

print("=" * 60)
print(f"SEARCH: {TEST_CATEGORY}  (Tier 1 + key Tier 2 only)")
print(f"Primary aliases: {FOCUSED_ALIASES}")
print(f"Secondary aliases (Tier 1 only): {SECONDARY_ALIASES}")
print("=" * 60)

# Pass 0: Analyst hub URLs
print("\n── Pass 0: Analyst author-hub URLs ──")
hub_added = 0
for hub_url in ANALYST_HUB_URLS:
    if hub_url not in seen_urls and not url_is_blocked(hub_url):
        seen_urls.add(hub_url)
        all_results.append({
            "url": hub_url,
            "title": f"[Hub] {hub_url.split('/')[-2] if hub_url.endswith('/') else hub_url.split('/')[-1]}",
            "snippet": "",
            "query_alias": TEST_CATEGORY,
            "search_pass": "AnalystHub",
        })
        hub_added += 1
print(f"  AnalystHub: {hub_added} direct URLs added")

# Pass 1: Tier 1 major analysts — primary aliases
print("\n── Pass 1: Tier 1 (Gartner, Forrester, IDC) — primary aliases ──")
total_queries += run_search_pass(
    "Tier1-primary", TIER1_SITES, FOCUSED_ALIASES,
    batch_size=3, seen=seen_urls, results=all_results, num_per_query=6
)

# Pass 1b: Tier 1 major analysts — secondary aliases
print("\n── Pass 1b: Tier 1 — secondary aliases ──")
total_queries += run_search_pass(
    "Tier1-secondary", TIER1_SITES, SECONDARY_ALIASES,
    batch_size=3, seen=seen_urls, results=all_results, num_per_query=6
)

# Pass 2: Key Tier 2 analysts — primary aliases only
print("\n── Pass 2: Key Tier 2 (Constellation, ISG, GigaOm, Nucleus, Aragon) ──")
total_queries += run_search_pass(
    "Tier2-key", KEY_TIER2_SITES, FOCUSED_ALIASES,
    batch_size=5, seen=seen_urls, results=all_results, num_per_query=6
)

# Summary
print(f"\n{'=' * 60}")
print(f"TOTAL: {total_queries} Serper queries → {len(all_results)} unique URLs")
print(f"{'=' * 60}")

from collections import Counter
pass_counts = Counter(r["search_pass"] for r in all_results)
for pass_name, count in pass_counts.items():
    print(f"  {pass_name}: {count} URLs")

print(f"\nAll {len(all_results)} results:")
for i, r in enumerate(all_results):
    print(f"  {i+1}. [{r['search_pass']}] {r['title'][:70]}")
    print(f"     {r['url']}")

cache_stats()

SEARCH: Account-Based Marketing  (Tier 1 + key Tier 2 only)
Aliases: ['Account-Based Marketing', 'ABM platforms']

── Pass 0: Analyst author-hub URLs ──
  AnalystHub: 6 direct URLs added

── Pass 1: Tier 1 (Gartner, Forrester, IDC) ──
  Tier1: 6 queries → 18 new URLs (blocked 1)

── Pass 2: Key Tier 2 (Constellation, ISG, GigaOm, Nucleus, Aragon) ──
  Tier2-key: 2 queries → 5 new URLs (blocked 0)

TOTAL: 8 Serper queries → 29 unique URLs
  AnalystHub: 6 URLs
  Tier1: 18 URLs
  Tier2-key: 5 URLs

All 29 results:
  1. [AnalystHub] [Hub] john_arnold
     https://www.forrester.com/blogs/author/john_arnold/
  2. [AnalystHub] [Hub] jessie_johnson
     https://www.forrester.com/blogs/author/jessie_johnson/
  3. [AnalystHub] [Hub] terry_flaherty
     https://www.forrester.com/blogs/author/terry_flaherty/
  4. [AnalystHub] [Hub] the-account-based-everything-framework
     https://www.gartner.com/en/articles/the-account-based-everything-framework
  5. [AnalystHub] [Hub] intelligent-marketing
   

In [6]:
# Cell 5: Scrape discovered URLs with Trafilatura
# Extracts: clean text, title, author, date — strips boilerplate automatically
import trafilatura
import time as _time

SCRAPE_DELAY = 0.5  # seconds between fetches to be polite

def extract_article(url: str) -> dict | None:
    """Download and extract article content from a URL.
    Returns dict with metadata or None on failure. Logs failure reason."""
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return {"_error": "fetch returned empty (403/timeout/JS-only)"}
        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=False,
            output_format="txt",
        )
        if not text:
            return {"_error": "extraction returned no text (template page?)"}
        meta = trafilatura.metadata.extract_metadata(downloaded)
        return {
            "url": url,
            "title": meta.title if meta else "",
            "author": meta.author if meta else "",
            "date": meta.date if meta else "",
            "text": text,
            "hostname": meta.sitename if meta else "",
        }
    except Exception as e:
        return {"_error": f"exception: {type(e).__name__}: {e}"}

# Scrape all discovered URLs
scraped_sources = []
scrape_failures = []
for i, r in enumerate(all_results):
    print(f"[{i+1}/{len(all_results)}] {r['url'][:80]}…", end=" ")
    article = extract_article(r["url"])
    if article and "_error" not in article and article["text"] and len(article["text"]) > 100:
        article["query_alias"] = r["query_alias"]
        article["search_pass"] = r["search_pass"]
        scraped_sources.append(article)
        print(f"✓ ({len(article['text'])} chars)")
    else:
        reason = article.get("_error", "too short") if article else "returned None"
        scrape_failures.append({"url": r["url"], "reason": reason})
        print(f"✗ ({reason})")
    _time.sleep(SCRAPE_DELAY)

print(f"\n{'='*50}")
print(f"Scraped {len(scraped_sources)} usable sources out of {len(all_results)} URLs")
print(f"Failures: {len(scrape_failures)}")
if scrape_failures:
    print("\nFailed URLs:")
    for f in scrape_failures[:10]:
        print(f"  ✗ {f['url'][:80]}… — {f['reason']}")

[1/29] https://www.forrester.com/blogs/author/john_arnold/… ✗ (fetch returned empty (403/timeout/JS-only))
[2/29] https://www.forrester.com/blogs/author/jessie_johnson/… ✓ (7573 chars)
[3/29] https://www.forrester.com/blogs/author/terry_flaherty/… ✓ (5853 chars)
[4/29] https://www.gartner.com/en/articles/the-account-based-everything-framework… ✗ (fetch returned empty (403/timeout/JS-only))
[5/29] https://research.isg-one.com/analyst-perspectives/topic/intelligent-marketing… ✓ (408 chars)
[6/29] https://chiefmartec.com/category/account-based-marketing/… ✗ (fetch returned empty (403/timeout/JS-only))
[7/29] https://www.forrester.com/blogs/what-is-account-based-marketing/… ✓ (7884 chars)
[8/29] https://www.forrester.com/blogs/category/account-based-marketing-abm/… ✓ (6132 chars)
[9/29] https://www.forrester.com/report/distributing-responsibilities-between-an-accoun… ✓ (670 chars)
[10/29] https://www.forrester.com/blogs/are-you-ready-for-abm/… ✓ (2732 chars)
[11/29] https://www.forrester.c

In [7]:
# Cell 6: Preview scraped sources
for i, s in enumerate(scraped_sources):
    print(f"--- Source {i+1} ---")
    print(f"  Title  : {s['title']}")
    print(f"  Author : {s['author']}")
    print(f"  Date   : {s['date']}")
    print(f"  Host   : {s['hostname']}")
    print(f"  Alias  : {s['query_alias']}")
    print(f"  Length  : {len(s['text'])} chars")
    print(f"  Preview: {s['text']}…")
    print()

--- Source 1 ---
  Title  : Jessie Johnson
  Author : Jessie Johnson
  Date   : 2026-03-10
  Host   : Forrester
  Alias  : Account-Based Marketing
  Length  : 7573 chars
  Preview: Jessie Johnson
Principal Analyst
Author Insights
Blog
The Future Of B2B GTM Isn’t Human Versus AI
AI has long been embedded in the B2B tech stack and go-to-market workflows. The sudden ubiquity of generative AI and, now, the promise of agentic AI have catalyzed rapid, intensive change — and driven the need for a harmonious human-AI coexistence.
Blog
Forrester Analyst Takes For Digital Content In 2026
Digital content will continue to be shaped by enterprises both investing in and looking to gain practical business value from genAI solutions. Here’s how we foresee digital content evolving.
Hear more from Jessie Johnson
Upcoming Events
B2B Summit North America
Upcoming Webinars
Check back soon for upcoming webinars.OnDemand Webinars
Check back soon for on-demand webinarsBlog
Personalization Makes Every Postsale

In [8]:
# Cell 6b: Deduplicate near-identical scraped content
# Many sites (e.g. Info-Tech SoftwareReviews comparisons) return identical boilerplate.
# Dedup by hashing the first 500 chars of extracted text.
import hashlib as _hl

def _content_hash(text: str) -> str:
    return _hl.md5(text[:500].strip().lower().encode()).hexdigest()

_seen_hashes = set()
deduped_sources = []
dupes_removed = 0
for s in scraped_sources:
    h = _content_hash(s["text"])
    if h in _seen_hashes:
        dupes_removed += 1
        continue
    _seen_hashes.add(h)
    deduped_sources.append(s)

print(f"Before dedup: {len(scraped_sources)} sources")
print(f"Duplicates removed: {dupes_removed}")
print(f"After dedup: {len(deduped_sources)} unique sources")

# Replace scraped_sources so downstream cells use deduped list
scraped_sources = deduped_sources

Before dedup: 26 sources
Duplicates removed: 0
After dedup: 26 unique sources


In [9]:
# Cell 7: Pre-filter — DROP_URL_PATTERNS + content-level quality gates
# Applies both URL-pattern blocking and minimum content thresholds.
from datetime import datetime, timedelta
from urllib.parse import urlparse

def source_age_months(date_str: str) -> int | None:
    """Estimate source age in months from a date string. Returns None if unparseable."""
    if not date_str:
        return None
    try:
        dt = datetime.strptime(date_str[:10], "%Y-%m-%d")
        delta = datetime.now() - dt
        return int(delta.days / 30.44)
    except (ValueError, TypeError):
        return None

def should_keep(source: dict) -> tuple[bool, str]:
    """Return (keep, reason) for a scraped source."""
    url = source["url"].lower()
    # URL-pattern drop
    for pattern in DROP_URL_PATTERNS:
        if pattern in url:
            return False, f"DROP pattern: {pattern}"
    # Minimum content length
    if len(source["text"]) < 300:
        return False, f"too short ({len(source['text'])} chars)"
    # Currency check — warn but don't drop (downstream scoring handles it)
    age = source_age_months(source.get("date"))
    if age is not None and age > MAX_SOURCE_AGE_MONTHS * 1.5:
        return False, f"too old ({age} months, threshold {MAX_SOURCE_AGE_MONTHS})"
    return True, "ok"

filtered_sources = []
dropped_sources = []
for s in scraped_sources:
    keep, reason = should_keep(s)
    if keep:
        filtered_sources.append(s)
    else:
        dropped_sources.append({"title": s.get("title", "?"), "url": s["url"], "reason": reason})

print(f"Kept {len(filtered_sources)} sources, dropped {len(dropped_sources)}")
if dropped_sources:
    print("\nDropped:")
    for d in dropped_sources:
        print(f"  ✗ {d['reason']}: {d['title'][:60]}")
        print(f"    {d['url']}")

print(f"\nFiltered sources:")
for i, s in enumerate(filtered_sources):
    age = source_age_months(s.get("date"))
    age_str = f"{age}mo" if age is not None else "?"
    print(f"  {i+1}. [{s.get('search_pass', '?')}] {s['title']}")
    print(f"     Author: {s['author'] or 'n/a'} | Date: {s['date'] or 'n/a'} ({age_str}) | {len(s['text'])} chars")
    print(f"     {s['url']}")
    print()

Kept 16 sources, dropped 10

Dropped:
  ✗ too old (96 months, threshold 24): Account-Based Marketing (ABM): The Ultimate SiriusDecisions 
    https://www.forrester.com/blogs/what-is-account-based-marketing/
  ✗ too old (92 months, threshold 24): Are You Ready for Account-Based Marketing?
    https://www.forrester.com/blogs/are-you-ready-for-abm/
  ✗ too old (49 months, threshold 24): The Forrester New Wave™: Account-Based Marketing Platforms, 
    https://www.forrester.com/report/the-forrester-new-wave-tm-account-based-marketing-platforms-q1-2022/RES176331
  ✗ too old (49 months, threshold 24): How ABM Technology Is Evolving The Modern Marketing Landscap
    https://www.forrester.com/blogs/how-abm-technology-is-evolving-the-modern-marketing-landscape/
  ✗ too old (70 months, threshold 24): The Second Forrester New Wave™ On ABM Platform Shows A Matur
    https://www.forrester.com/blogs/second-abm-platform-new-wave-shows-a-maturing-market/
  ✗ too old (95 months, threshold 24): The Forre

In [10]:
# Cell 8: LLM-based source quality scoring (per doc1 §5)
# Scores each source on: slot-fill, function-verbs, author credibility, currency, vendor diversity
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import json, time as _time

LLM_DELAY = 0.2  # seconds between LLM calls

class SourceScore(BaseModel):
    slot_definition: bool = Field(description="Contains a category definition")
    slot_capabilities: bool = Field(description="Lists core capabilities of the software")
    slot_boundaries: bool = Field(description="Distinguishes from adjacent/related categories")
    slot_buyer_use: bool = Field(description="Describes buyer persona or use case")
    slot_vendors: bool = Field(description="Names representative vendors/products")
    slots_filled: int = Field(description="Count of slots filled (0-5)")
    uses_function_verbs: bool = Field(description="Uses expert verbs like orchestrate, unify, score, route, match (vs SEO adjectives like better, smarter, faster)")
    vendor_count: int = Field(description="Number of distinct vendors/products mentioned")
    vendor_names: list[str] = Field(description="List of distinct vendor/product names found")
    is_sme_content: bool = Field(description="Appears to be written by or for subject-matter experts, not SEO/marketing fluff")
    byline_quality: str = Field(description="'named_analyst' if byline is a recognized analyst at a known firm, 'named_author' if any named author, 'no_byline' if anonymous/staff")
    single_vendor_bias: bool = Field(description="True if source primarily promotes a single vendor rather than providing neutral analysis")
    relevance_score: int = Field(description="1-10 overall relevance to defining the software category")
    reasoning: str = Field(description="Brief explanation of the score")

SCORING_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are evaluating a web source for its usefulness in defining the software category "{category}".

Score it on these criteria (doc1 §5):

1. SLOT-FILL: Does it contain (a) a definition, (b) core capabilities, (c) boundaries vs adjacent categories, (d) buyer/use case, (e) representative vendors? Count how many of these 5 slots it fills.

2. FUNCTION-VERBS: Does it use expert verbs (orchestrate, unify, score, route, match, segment, personalize, align) rather than SEO adjectives (better, smarter, faster, top, best)? This is one of the cleanest SME vs fluff filters.

3. BYLINE QUALITY: Evaluate the author byline.
   - "named_analyst" = recognized analyst at a known firm (Gartner, Forrester, Constellation, ISG, etc.)
   - "named_author" = has a named author but not clearly a recognized analyst
   - "no_byline" = anonymous, staff-writer, or no author attribution
   For trade publications (CIO, ZDNet, Diginomica, MarTech, etc.), only "named_analyst" or recognized SME bylines make the source trustworthy. Staff-writer SEO pieces should score lower.

4. VENDOR DIVERSITY & BIAS: How many distinct vendors are named? List them. A source naming 5+ vendors from different corporate families shows range. A source naming only its host vendor or a tight 2-3 from the same family signals selection bias. Flag single_vendor_bias if the source primarily promotes one vendor.

5. SME CONTENT: Is this analyst/expert content or marketing fluff? Consider the depth of analysis, presence of frameworks, and whether it reads as editorial work vs promotional material.

6. RELEVANCE: How useful is this source for writing a definitive category page (1-10)?
   - 8-10: Fills 4-5 slots, uses function-verbs, SME-authored, multi-vendor
   - 5-7: Fills 2-3 slots or has partial quality signals
   - 1-4: Off-topic, thin, or promotional

Return valid JSON matching the schema."""),
    ("human", """Source URL: {url}
Title: {title}
Author: {author}
Date: {date}
Host: {hostname}
Source age: {source_age}

Content (first 3000 chars):
{text}"""),
])

# Print the full prompt template once for review
print("SCORING PROMPT TEMPLATE:")
print("=" * 60)
for msg in SCORING_PROMPT.messages:
    print(f"\n── {msg.prompt.template[:20]}… ({type(msg).__name__}) ──")
print(f"\nInput variables: {SCORING_PROMPT.input_variables}")
print("=" * 60)

llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0, api_key=OPENAI_API_KEY)
structured_llm = llm.with_structured_output(SourceScore)
chain = SCORING_PROMPT | structured_llm

scored_sources = []
cache_hits_llm = 0
for i, s in enumerate(filtered_sources):
    print(f"[{i+1}/{len(filtered_sources)}] Scoring: {s['title'][:60]}…", end=" ")
    text_trunc = s["text"][:3000]
    age = source_age_months(s.get("date"))
    age_str = f"{age} months" if age is not None else "unknown"

    invoke_params = {
        "category": TEST_CATEGORY,
        "url": s["url"],
        "title": s["title"] or "Unknown",
        "author": s["author"] or "Unknown",
        "date": s["date"] or "Unknown",
        "hostname": s["hostname"] or "Unknown",
        "source_age": age_str,
        "text": text_trunc,
    }
    cache_key_parts = ("scoring_v2", TEST_CATEGORY, s["url"], s["title"] or "Unknown",
                       s["author"] or "Unknown", s["date"] or "Unknown",
                       s["hostname"] or "Unknown", text_trunc)
    cached = cache_get("llm_scoring", *cache_key_parts)
    if cached is not None:
        score_data = cached["score"] if "score" in cached else cached
        score = SourceScore(**score_data)
        cache_hits_llm += 1
        scored_sources.append({"source": s, "score": score})
        print(f"✓ relevance={score.relevance_score}/10, slots={score.slots_filled}/5 (cached)")
        continue
    try:
        score = chain.invoke(invoke_params)
        # Cache both score and rendered prompt
        rendered_msgs = SCORING_PROMPT.format_messages(**invoke_params)
        rendered_prompt = "\n".join(f"[{m.type}] {m.content}" for m in rendered_msgs)
        cache_set("llm_scoring", {
            "score": score.model_dump(),
            "prompt": rendered_prompt,
        }, *cache_key_parts)
        scored_sources.append({"source": s, "score": score})
        print(f"✓ relevance={score.relevance_score}/10, slots={score.slots_filled}/5")
        _time.sleep(LLM_DELAY)
    except Exception as e:
        print(f"✗ {e}")

# Sort by relevance score descending
scored_sources.sort(key=lambda x: x["score"].relevance_score, reverse=True)
print(f"\nScored {len(scored_sources)} sources. Top sources:")
if cache_hits_llm:
    print(f"  ({cache_hits_llm} scores loaded from cache, {len(scored_sources) - cache_hits_llm} new LLM calls)")
cache_stats()

c:\Users\Abhishek A\Defining_Category\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SCORING PROMPT TEMPLATE:

── You are evaluating a… (SystemMessagePromptTemplate) ──

── Source URL: {url}
Ti… (HumanMessagePromptTemplate) ──

Input variables: ['author', 'category', 'date', 'hostname', 'source_age', 'text', 'title', 'url']
[1/16] Scoring: Jessie Johnson… ✓ relevance=2/10, slots=0/5
[2/16] Scoring: Terry Flaherty… ✓ relevance=2/10, slots=0/5
[3/16] Scoring: ISG Software Research Analyst Perspectives | intelligent mar… ✓ relevance=2/10, slots=0/5
[4/16] Scoring: Account-Based Marketing (ABM) Archives… ✓ relevance=4/10, slots=1/5
[5/16] Scoring: Distributing Responsibilities Between An Account-Based Marke… ✓ relevance=8/10, slots=4/5
[6/16] Scoring: Establishing An Account-Based Marketing Charter | Forrester… ✓ relevance=8/10, slots=4/5
[7/16] Scoring: How ABM advertising accelerates B2B growth… ✓ relevance=9/10, slots=5/5
[8/16] Scoring: Accelerate Account-Based Marketing: Orient Your Strategy wit… ✓ relevance=8/10, slots=4/5
[9/16] Scoring: Accelerate Account-Based Mar

In [11]:
# Cell 9: Display scored sources ranked by relevance
for i, item in enumerate(scored_sources):
    s = item["source"]
    sc = item["score"]
    print(f"{'='*60}")
    print(f"#{i+1}  Relevance: {sc.relevance_score}/10 | Slots: {sc.slots_filled}/5 | Vendors: {sc.vendor_count}")
    print(f"  Title : {s['title']}")
    print(f"  Author: {s['author'] or 'n/a'} | Date: {s['date'] or 'n/a'} | Host: {s['hostname'] or 'n/a'}")
    print(f"  URL   : {s['url']}")
    print(f"  Byline: {sc.byline_quality} | Expert verbs: {sc.uses_function_verbs} | SME: {sc.is_sme_content} | Bias: {sc.single_vendor_bias}")
    print(f"  Slots → Def:{sc.slot_definition} Cap:{sc.slot_capabilities} Bound:{sc.slot_boundaries} Buyer:{sc.slot_buyer_use} Vendors:{sc.slot_vendors}")
    if sc.vendor_names:
        print(f"  Vendors named: {', '.join(sc.vendor_names)}")
    print(f"  Reasoning: {sc.reasoning}")
    print()

# Select top sources for synthesis (relevance >= 5 and at least 2 slots filled)
# Also exclude single-vendor-bias sources unless they're the only option
top_sources = [
    item for item in scored_sources
    if item["score"].relevance_score >= 5
    and item["score"].slots_filled >= 2
    and not item["score"].single_vendor_bias
]
# If too few, relax the bias filter
if len(top_sources) < 3:
    top_sources = [
        item for item in scored_sources
        if item["score"].relevance_score >= 5
        and item["score"].slots_filled >= 2
    ]

print(f"\n{'='*60}")
print(f"Sources qualifying for synthesis: {len(top_sources)} (relevance≥5, slots≥2, no single-vendor bias)")
if len(top_sources) < 3:
    print("⚠️  WARNING: Fewer than 3 qualifying sources — synthesis may be thin.")

#1  Relevance: 10/10 | Slots: 5/5 | Vendors: 12
  Title : Converging Platforms For Greater Efficiency: The Rise Of Revenue Marketing Platforms
  Author: Kelvin Gee | Date: 2024-07-25 | Host: Forrester
  URL   : https://www.forrester.com/blogs/converging-platforms-for-greater-efficiency-the-rise-of-revenue-marketing-platforms/
  Byline: named_analyst | Expert verbs: True | SME: True | Bias: False
  Slots → Def:True Cap:True Bound:True Buyer:True Vendors:True
  Vendors named: Vendor 1, Vendor 2, Vendor 3, Vendor 4, Vendor 5, Vendor 6, Vendor 7, Vendor 8, Vendor 9, Vendor 10, Vendor 11, Vendor 12
  Reasoning: The source provides a comprehensive definition of revenue marketing platforms, lists core capabilities, distinguishes it from adjacent categories like MAPs, describes buyer use cases, and names multiple vendors. It uses expert verbs throughout and is authored by a recognized analyst at Forrester, indicating high-quality SME content. Overall, it is highly relevant for defining the sof

In [12]:
# Cell 10: Multi-source synthesis — produce the 5-slot category page (per doc1 §6)
# Uses LangChain structured output: feed all top sources into one synthesis call.
# NOTE: For production quality, consider using gpt-4o instead of gpt-4o-mini.
# Set OPENAI_MODEL=gpt-4o in .env for deeper reasoning on complex categories.

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class CategoryPage(BaseModel):
    category_name: str = Field(description="Primary category name")
    aliases: list[str] = Field(description="Known aliases for this category")
    definition: str = Field(description="2-4 sentence category definition synthesized from multiple sources")
    core_capabilities: list[str] = Field(description="Core software capabilities (use function-verbs: orchestrate, unify, score, route, segment, personalize)")
    boundaries: str = Field(description="What this category is NOT; how it differs from adjacent categories. Name specific adjacent categories.")
    buyer_use_case: str = Field(description="Who buys this software and why — be specific about roles and organization types")
    representative_vendors: list[str] = Field(description="Named vendors from across sources — must come from multiple sources, not a single source")
    category_drift: str = Field(description="Where analyst firms disagree on scope, naming, or existence. Name the specific firms and their positions. Empty string if no disagreement found.")
    source_count: int = Field(description="Number of sources used in synthesis")
    confidence: str = Field(description="high/medium/low — based on source coverage and consensus")

SYNTHESIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are writing a category definition page for Cuspera. The page must be written in Cuspera's editorial voice — no direct quotation from sources (copyright). Multi-source consensus carries the definition.

RULES (from doc1 §6):
- Fill five slots in order: definition, core capabilities, boundaries, buyer/use case, representative vendors.
- DEFINITION: 2-4 sentences. Must describe the SOFTWARE category, not the methodology. What the software does, not what the strategy is.
- CORE CAPABILITIES: Use function-verbs (orchestrate, unify, score, route, segment, personalize, align, prioritize, automate, enrich, match). Describe what the SOFTWARE does. Do NOT use benefit-adjectives (better, smarter, faster, top, best).
- BOUNDARIES: Name specific adjacent categories this is NOT — e.g. "ABM platforms are distinct from general Marketing Automation platforms (Marketo, HubSpot MA) which focus on lead-level rather than account-level engagement." Be concrete.
- BUYER/USE CASE: Name specific buyer roles (CMO, VP Demand Gen, Revenue Marketing) and organization types (B2B enterprise, mid-market SaaS).
- VENDORS: Must come from across multiple sources, not a single source. If a vendor appears in only one source, note that.
- After the slots, address category drift if applicable. Name the specific firms: "Forrester frames this as X while Gartner uses the term Y." Do not use vague language like "some firms."
- If analyst firms genuinely disagree on whether this is a distinct software market, acknowledge it directly.

Category: {category}
Maturity: {maturity}
Aliases: {aliases}"""),
    ("human", """Here are the top-scoring sources to synthesize from:

{sources_text}

Synthesize these into a single category page. Return structured JSON."""),
])

# Prepare source texts for the prompt
sources_block = ""
for i, item in enumerate(top_sources):
    s = item["source"]
    sc = item["score"]
    sources_block += f"\n--- SOURCE {i+1} (relevance {sc.relevance_score}/10, slots {sc.slots_filled}/5) ---\n"
    sources_block += f"Title: {s['title']}\n"
    sources_block += f"Author: {s['author'] or 'Unknown'} | Host: {s['hostname'] or 'Unknown'} | Date: {s['date'] or 'Unknown'}\n"
    sources_block += f"Byline quality: {sc.byline_quality} | Vendors named: {', '.join(sc.vendor_names) if sc.vendor_names else 'none'}\n"
    sources_block += f"Content:\n{s['text'][:4000]}\n"

print(f"Synthesizing from {len(top_sources)} sources ({len(sources_block)} chars total)…\n")

# Check synthesis cache
synth_cache_key = ("synthesis_v2", TEST_CATEGORY, ", ".join(CATEGORY_ALIASES), sources_block)
cached_synth = cache_get("llm_synthesis", *synth_cache_key)
if cached_synth is not None:
    category_page = CategoryPage(**cached_synth)
    print("✓ Synthesis loaded from cache!")
else:
    synth_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2, api_key=OPENAI_API_KEY)
    synth_chain = SYNTHESIS_PROMPT | synth_llm.with_structured_output(CategoryPage)

    category_page = synth_chain.invoke({
        "category": TEST_CATEGORY,
        "maturity": CATEGORY_MATURITY,
        "aliases": ", ".join(CATEGORY_ALIASES),
        "sources_text": sources_block,
    })
    # Cache the result
    cache_set("llm_synthesis", category_page.model_dump(), *synth_cache_key)
    print("✓ Synthesis complete (cached for next run)!")

cache_stats()

Synthesizing from 7 sources (23063 chars total)…

✓ Synthesis complete (cached for next run)!
  llm_scoring: 16 entries
  llm_synthesis: 1 entries
  serper: 8 entries


In [13]:
# Cell 11: Display the final synthesized category page + export to JSON
import json, pathlib

print(f"{'='*70}")
print(f"  CATEGORY PAGE: {category_page.category_name}")
print(f"{'='*70}")
print(f"\nAliases: {', '.join(category_page.aliases)}")
print(f"Confidence: {category_page.confidence} | Sources used: {category_page.source_count}")

print(f"\n{'─'*70}")
print("1. DEFINITION")
print(f"{'─'*70}")
print(category_page.definition)

print(f"\n{'─'*70}")
print("2. CORE CAPABILITIES")
print(f"{'─'*70}")
for cap in category_page.core_capabilities:
    print(f"  • {cap}")

print(f"\n{'─'*70}")
print("3. BOUNDARIES")
print(f"{'─'*70}")
print(category_page.boundaries)

print(f"\n{'─'*70}")
print("4. BUYER / USE CASE")
print(f"{'─'*70}")
print(category_page.buyer_use_case)

print(f"\n{'─'*70}")
print("5. REPRESENTATIVE VENDORS")
print(f"{'─'*70}")
for v in category_page.representative_vendors:
    print(f"  • {v}")

if category_page.category_drift:
    print(f"\n{'─'*70}")
    print("6. CATEGORY DRIFT / ANALYST DISAGREEMENT")
    print(f"{'─'*70}")
    print(category_page.category_drift)

print(f"\n{'='*70}")
print("Sources used:")
for i, item in enumerate(top_sources):
    s = item["source"]
    sc = item["score"]
    print(f"  [{i+1}] {s['title']} — {s['author'] or 'n/a'} ({s['hostname'] or 'n/a'})")
    print(f"      {s['url']}  [relevance {sc.relevance_score}/10, byline: {sc.byline_quality}]")

# ── Export results to JSON for persistence ──────────────────────────────
output_dir = pathlib.Path("output")
output_dir.mkdir(exist_ok=True)

# Export category page
cat_slug = TEST_CATEGORY.lower().replace(" ", "_").replace("-", "_")
page_path = output_dir / f"{cat_slug}_page.json"
page_path.write_text(json.dumps(category_page.model_dump(), indent=2, ensure_ascii=False), encoding="utf-8")

# Export scored sources (for audit trail)
scores_path = output_dir / f"{cat_slug}_scores.json"
scores_export = []
for item in scored_sources:
    s = item["source"]
    sc = item["score"]
    scores_export.append({
        "url": s["url"],
        "title": s["title"],
        "author": s["author"],
        "date": s["date"],
        "hostname": s["hostname"],
        "search_pass": s.get("search_pass", ""),
        "text_length": len(s["text"]),
        "score": sc.model_dump(),
    })
scores_path.write_text(json.dumps(scores_export, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"\n{'='*70}")
print(f"Exported:")
print(f"  Category page → {page_path}")
print(f"  Scored sources ({len(scores_export)}) → {scores_path}")

  CATEGORY PAGE: Account-Based Marketing

Aliases: Account-Based Marketing, ABM, Account-Based Marketing Platforms, ABM platforms, Account-Based Everything, ABX, Account-Based Experience
Confidence: high | Sources used: 7

──────────────────────────────────────────────────────────────────────
1. DEFINITION
──────────────────────────────────────────────────────────────────────
Account-Based Marketing (ABM) is a strategic approach in B2B marketing that focuses on targeting specific accounts or buying groups rather than individual leads. This software category enables organizations to design and execute highly personalized marketing campaigns tailored to the unique needs and characteristics of identified accounts, facilitating deeper engagement and alignment between sales and marketing efforts.

──────────────────────────────────────────────────────────────────────
2. CORE CAPABILITIES
──────────────────────────────────────────────────────────────────────
  • orchestrate
  • personalize
 

In [14]:
# Cell 12: Second-pass review placeholder (per doc1 §1)
# "Two passes per category. First pass casts the net wide...
#  Second pass to check for derivative sources, and understand what couldn't be pinned down."
#
# TODO: Implement second pass:
# 1. Review the category_page output for gaps (e.g. missing vendors, weak boundaries)
# 2. Generate targeted follow-up queries for specific gaps
# 3. Check if any sources are derivative (citing the same underlying analyst report)
# 4. Flag what couldn't be pinned down for manual review
#
# For now, print a gap analysis based on the synthesis output.

gaps = []
if category_page.confidence != "high":
    gaps.append(f"Confidence is '{category_page.confidence}' — may need more sources")
if len(category_page.representative_vendors) < 5:
    gaps.append(f"Only {len(category_page.representative_vendors)} vendors — aim for 5+")
if not category_page.category_drift:
    gaps.append("No category drift detected — verify manually that analysts agree")
if len(category_page.core_capabilities) < 5:
    gaps.append(f"Only {len(category_page.core_capabilities)} capabilities listed — may be incomplete")
if len(top_sources) < 5:
    gaps.append(f"Only {len(top_sources)} qualifying sources — thin evidence base")

# Check source diversity
source_hosts = set()
for item in top_sources:
    host = item["source"].get("hostname", "")
    if host:
        source_hosts.add(host.lower().split("|")[0].strip())
if len(source_hosts) < 3:
    gaps.append(f"Sources from only {len(source_hosts)} distinct hosts — low diversity")

# Check for Tier 1 presence
tier1_found = any(
    any(t1 in item["source"]["url"] for t1 in ["forrester.com", "gartner.com", "idc.com"])
    for item in top_sources
)
if not tier1_found:
    gaps.append("⚠️  No Tier 1 analyst sources (Gartner/Forrester/IDC) in synthesis — major gap")

print(f"{'='*60}")
print("SECOND-PASS GAP ANALYSIS")
print(f"{'='*60}")
if gaps:
    for g in gaps:
        print(f"  ⚠️  {g}")
    print(f"\n{len(gaps)} gaps identified — consider targeted follow-up searches.")
else:
    print("  ✓ No major gaps detected.")
print(f"\nSource host diversity: {', '.join(sorted(source_hosts))}")

SECOND-PASS GAP ANALYSIS
  ⚠️  Sources from only 2 distinct hosts — low diversity

1 gaps identified — consider targeted follow-up searches.

Source host diversity: forrester, idc
